# Parameterized circuits and feature-map templates

**목표.** Qiskit v2.x 자격증 시험에 필요한 최소한의 parameterized circuit 사용법을 익힌다.  
핵심은 `Parameter`, `ParameterVector`, `assign_parameters`, reusable template, 그리고 feature-map circuit을 구분하는 것이다.


## 요약 대상 source notebook

| Source notebook | 중요하게 볼 내용 | 가볍게 볼 내용 |
|---|---|---|
| `2_Construct_Parameterized_Circuits.ipynb` | `Parameter`, `ParameterVector`, `assign_parameters`, template 재사용 | 긴 parameter sweep 예제 |
| `2_3_Quantum_Operations_FeatureMaps.ipynb` | feature map이 classical data를 quantum circuit으로 encoding하는 방식 | 출력이 많은 visualization |
| `1_Set_Estimator_Options.ipynb` | parameterized circuit을 Estimator workflow에 넣는 흐름 | option 세부값 |

**핵심 아이디어.** Parameterized circuit은 numerical value를 나중에 넣을 수 있는 circuit template이다. Feature map은 classical data vector를 circuit parameter에 넣어 quantum state로 encoding하는 template이다.


In [1]:
from math import pi
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector


## 1. `Parameter`와 `ParameterExpression`

`Parameter`는 circuit을 만들 때 쓰는 symbolic variable이다.  
숫자를 넣을 때는 `assign_parameters()`를 사용한다.


In [2]:
theta = Parameter("θ")

qc = QuantumCircuit(1)
qc.rx(theta, 0)

print("before binding:", qc.parameters)

bound_qc = qc.assign_parameters({theta: pi / 4})
print("after binding:", bound_qc.parameters)
bound_qc.draw("text")


before binding: ParameterView([Parameter(θ)])
after binding: ParameterView([])


┌─────────┐
q: ┤ Rx(π/4) ├
   └─────────┘

## 2. `ParameterVector`

여러 parameter가 필요할 때는 `ParameterVector`를 쓰면 편하다.  
예를 들어 variational ansatz나 feature map에서 자주 사용한다.


In [3]:
params = ParameterVector("θ", 3)

qc = QuantumCircuit(3)
for i in range(3):
    qc.ry(params[i], i)

print(qc.parameters)
qc.draw("text")


ParameterView([ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2])])


┌──────────┐
q_0: ┤ Ry(θ[0]) ├
     ├──────────┤
q_1: ┤ Ry(θ[1]) ├
     ├──────────┤
q_2: ┤ Ry(θ[2]) ├
     └──────────┘

## 3. Reusable parameterized template

Parameterized circuit은 같은 구조를 유지하면서 parameter 값만 바꿔 재사용할 수 있다.


In [4]:
def ry_cx_template(num_qubits: int) -> QuantumCircuit:
    theta = ParameterVector("θ", num_qubits)
    qc = QuantumCircuit(num_qubits, name="ry_cx_template")

    for i in range(num_qubits):
        qc.ry(theta[i], i)

    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)

    return qc

template = ry_cx_template(3)
values = {p: 0.1 * k for k, p in enumerate(template.parameters, start=1)}

bound = template.assign_parameters(values)
print("template parameters:", template.parameters)
print("bound parameters:", bound.parameters)
bound.draw("text")


template parameters: ParameterView([ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2])])
bound parameters: ParameterView([])


┌─────────┐          
q_0: ┤ Ry(0.1) ├──■───────
     ├─────────┤┌─┴─┐     
q_1: ┤ Ry(0.2) ├┤ X ├──■──
     ├─────────┤└───┘┌─┴─┐
q_2: ┤ Ry(0.3) ├─────┤ X ├
     └─────────┘     └───┘

## 4. Feature-map template

Feature map은 classical data를 circuit parameter로 넣어 quantum state에 encoding하는 circuit이다.  
Qiskit 2.x에서는 `ZZFeatureMap` class보다 `zz_feature_map()` function 사용을 권장한다.


In [5]:
from qiskit.circuit.library import zz_feature_map

feature_map = zz_feature_map(feature_dimension=3, reps=1, entanglement="linear")

x = [0.1, 0.2, 0.3]
bound_feature_map = feature_map.assign_parameters(x)

print("unbound parameters:", feature_map.parameters)
print("bound parameters:", bound_feature_map.parameters)
bound_feature_map.draw("text")


unbound parameters: ParameterView([ParameterVectorElement(x[0]), ParameterVectorElement(x[1]), ParameterVectorElement(x[2])])
bound parameters: ParameterView([])


┌───┐┌────────┐                                              
0: ┤ H ├┤ P(0.2) ├──■─────────────────■─────────────────────────
   ├───┤├────────┤┌─┴─┐┌───────────┐┌─┴─┐                       
1: ┤ H ├┤ P(0.4) ├┤ X ├┤ P(17.894) ├┤ X ├──■─────────────────■──
   ├───┤├────────┤└───┘└───────────┘└───┘┌─┴─┐┌───────────┐┌─┴─┐
2: ┤ H ├┤ P(0.6) ├───────────────────────┤ X ├┤ P(16.718) ├┤ X ├
   └───┘└────────┘                       └───┘└───────────┘└───┘

## 5. Parameterized circuit 확인하기

| 목적 | 표현 | 의미 |
|---|---|---|
| 남은 parameter 확인 | `qc.parameters` | 아직 값이 없는 symbolic parameter |
| 값 대입 | `qc.assign_parameters({...})` | parameter를 숫자 또는 다른 expression으로 치환 |
| 회로 구조 확인 | `qc.draw()` | circuit diagram 출력 |
| 연산 개수 확인 | `qc.count_ops()` | gate/instruction 종류별 개수 |
| 깊이 확인 | `qc.depth()` | circuit depth |
| template 재사용 | function returning `QuantumCircuit` | 같은 구조를 여러 번 생성 |
| feature map | `zz_feature_map(...)` | classical data를 quantum circuit parameter로 encoding |


## 자격증 시험 스타일 연습문제

각 문제에서 a), b), c), d) 중 하나를 고르시오.

1. `Parameter`의 가장 적절한 설명은 무엇인가?

```python
theta = Parameter("θ")
qc.rx(theta, 0)
```

a) 실행 중 측정값을 저장하는 classical bit이다.  
b) circuit을 만들 때 사용하는 compile-time symbolic variable이다.  
c) backend의 coupling map을 나타낸다.  
d) measurement 결과의 histogram이다.

2. parameterized circuit에 numerical value를 대입할 때 주로 사용하는 method는 무엇인가?

a) `measure_all()`  
b) `assign_parameters()`  
c) `count_ops()`  
d) `decompose()`

3. 아직 bind되지 않은 parameter를 확인하는 표현은 무엇인가?

a) `qc.parameters`  
b) `qc.clbits`  
c) `qc.cregs`  
d) `qc.global_phase`

4. 여러 개의 관련 parameter를 만들 때 가장 편한 객체는 무엇인가?

a) `QuantumRegister`  
b) `ClassicalRegister`  
c) `ParameterVector`  
d) `ControlledGate`

5. 다음 중 feature map의 목적에 가장 가까운 것은 무엇인가?

a) classical data를 quantum circuit parameter로 encoding한다.  
b) circuit을 backend basis gate로 transpile한다.  
c) measurement 결과를 histogram으로 그린다.  
d) qubit을 classical bit으로 변환한다.

6. Qiskit 2.x에서 `ZZFeatureMap` class 대신 권장되는 형태는 무엇인가?

a) `zz_feature_map()` function  
b) `QuantumRegister()`  
c) `Sampler()`  
d) `Measure()`

7. `assign_parameters()`의 기본 동작으로 가장 적절한 것은 무엇인가?

a) 원래 circuit을 항상 직접 변경한다.  
b) parameter가 남아 있으면 항상 error를 낸다.  
c) 보통 parameter가 대입된 새 circuit을 반환한다.  
d) circuit을 자동으로 측정한다.

8. parameterized template을 쓰는 주된 이유는 무엇인가?

a) 매번 완전히 다른 backend를 만들기 위해서  
b) 같은 circuit 구조를 유지하고 값만 바꿔 재사용하기 위해서  
c) measurement를 제거하기 위해서  
d) classical register를 없애기 위해서


## 간단한 정답 해설

1. **b)** `Parameter`는 circuit 생성 단계에서 쓰는 symbolic variable이다.  
2. **b)** numerical value 대입에는 `assign_parameters()`를 사용한다.  
3. **a)** `qc.parameters`는 아직 값이 없는 parameter set을 보여준다.  
4. **c)** `ParameterVector`는 여러 parameter를 한 번에 만들 때 유용하다.  
5. **a)** feature map은 classical input을 quantum state 준비 circuit에 넣는 역할을 한다.  
6. **a)** Qiskit 2.x에서는 `zz_feature_map()` function이 권장된다.  
7. **c)** 기본적으로 새 circuit을 반환하며, `inplace=True`를 쓰면 직접 변경할 수 있다.  
8. **b)** template 구조를 재사용하고 parameter 값만 바꿀 수 있다.
